# ModernBERT Fine-tuning (Sentiment Classification of Guardian Headlines)

ModernBERT is **encoder-only**: every token attends to every other token in both directions, so the
model builds one representation of the whole headline and a classification head reads a single label
off it. That is the right shape for classification and the wrong shape for generation — compare
`train_t5.ipynb` (encoder-decoder, rewrites text) and `train_gpt2.ipynb` (decoder-only, continues
text).

The Guardian file ships with no sentiment column, so the labels are **weak**:
`headlines.bert.utils.labeling` derives them from TextBlob polarity. That is a deterministic
heuristic, not human annotation, and it puts a ceiling on what any model here can score.

> **Before you run this:** switch on the GPU with **Runtime -> Change runtime type -> Hardware
> accelerator -> GPU**.

> **On precision:** `resolve_precision()` in `headlines/utils.py` asks for bf16 when the GPU supports
> it, because ModernBERT was pretrained in bf16 and can emit NaN losses in fp16. A **T4 has no bf16**,
> so it falls back to fp32 rather than risking the loss — slower, but it finishes. Setting `bf16=True`
> or `fp16=True` yourself overrides the whole decision.

In [ ]:
![ -d /content/Bert-T5-GPT2 ] || git clone https://github.com/nickkats1/Bert-T5-GPT2

## Install the package from the clone

The install has to be **editable** (`-e`). `headlines/config.py` computes

```python
PROJECT_ROOT = Path(__file__).resolve().parents[2]
```

so `GUARDIAN_PATH` and `REUTERS_PATH` are found relative to wherever `config.py` physically lives. An
editable install leaves it inside the clone next to `data/`; a regular install copies it into
`site-packages`, where `parents[2]` points at nothing useful and every default `data_path` breaks.

In [ ]:
%pip install -q -e /content/Bert-T5-GPT2

## Confirm the GPU is visible

`resolve_precision()` in `headlines/utils.py` reads exactly these two values. bf16 where the device
supports it, fp32 everywhere else — never fp16. ModernBERT and T5 were both pretrained in bf16 and
emit NaN losses under fp16, which does not raise: training runs to completion and the score comes
back at zero. A T4 has no bf16, so it trains in fp32 and takes longer.

In [ ]:
import torch


print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device:        {torch.cuda.get_device_name(0)}")
    print(f"bf16 support:  {torch.cuda.is_bf16_supported()}")

## Configure the run

Configuration is split three ways, the same split the `transformers` example scripts use.
`ClassificationModelArguments` says which checkpoint to fine-tune, `ClassificationDataArguments` says
what to train on, and `CLASSIFICATION` in `headlines/bert/config.py` is a plain dictionary of training
settings that `training_arguments()` turns into a real `TrainingArguments`.

Because training settings come from the library rather than being re-declared, they use the library's
spellings: `num_train_epochs`, not `epochs`; `per_device_train_batch_size`, not `batch_size`. Every
key in the dictionary is a real field, so a typo raises here rather than twenty minutes into a run.

In [ ]:
from headlines.bert.config import CLASSIFICATION, ClassificationDataArguments, ClassificationModelArguments
from headlines.config import training_arguments


model_args = ClassificationModelArguments()
data_args = ClassificationDataArguments()
training_args = training_arguments(CLASSIFICATION, output_dir="/content/artifacts/bert")

print(model_args)
print(data_args)

## Load and label the headlines

`load_csv` drops incomplete and duplicate rows, then `label_headlines` scores each headline with
`polarity` and buckets it through `sentiment`. Anything with exactly zero polarity lands in Neutral,
which is why that class ends up so large.

In [ ]:
from headlines.bert.utils.labeling import ID2LABEL, label_headlines
from headlines.data import load_csv


frame = label_headlines(load_csv(data_args.data_path, [data_args.text_column]), data_args.text_column)
print(f"{len(frame):,} headlines after cleaning")
frame.head(10)

### Class balance

TextBlob calls most news headlines neutral. That imbalance is the reason `score_predictions` reports
**macro** precision/recall/F1 next to the weighted versions: a model that predicts Neutral for
everything still scores well on the weighted numbers and badly on the macro ones.

In [ ]:
import matplotlib.pyplot as plt


counts = frame["labels"].map(ID2LABEL).value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_title("Weak sentiment labels from TextBlob polarity")
ax.set_ylabel("headlines")
ax.grid(True, axis="y", alpha=0.3)
plt.show()

## Tokenize and split

`build_datasets` holds out `data_args.test_size` and splits that holdout by
`data_args.val_test_ratio`, giving an 80/10/10 `train` / `validation` / `test` partition. The split is
stratified on the label and seeded with `training_args.seed`, so calling this here and letting
`build_trainer` call it again produces the identical partition. The second call is close to free —
`datasets.map` fingerprints its arguments and reuses the cached arrow file.


In [ ]:
from transformers import AutoTokenizer

from headlines.bert.dataset import build_datasets


tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path)
datasets = build_datasets(data_args, tokenizer, training_args.seed)

for name, split in datasets.items():
    print(f"{name:<11} {len(split):>7,} rows  columns={split.column_names}")

example = datasets["train"][0]
print()
print(tokenizer.decode(example["input_ids"]))
print(f"label: {ID2LABEL[example['labels']]}")

## Build the Trainer

`build_trainer` wires up the model, the padding collator, `compute_metrics`, and an
`EarlyStoppingCallback` with `training_args.early_stopping_patience`. Best-checkpoint selection is on
`f1_weighted`, and `load_best_model_at_end` means the object you hold after training is the best
epoch, not the last.

The head is sized from `len(LABEL_MAP)` rather than a separate `num_labels` setting, so the head and
the `id2label` mapping cannot drift apart.

Early stopping and checkpoint selection both read `validation`, so the per-epoch scores below are
optimistic by construction — the epoch was chosen because it looked good on exactly those rows.
`test` stays out of every one of those decisions and is scored once at the end.


In [ ]:
from headlines.bert.train import build_trainer


trainer = build_trainer(model_args, data_args, training_args)
print(f"training on {len(trainer.train_dataset):,} rows for {training_args.num_train_epochs:g} epochs")
print(f"precision: fp16={trainer.args.fp16} bf16={trainer.args.bf16}")

## Smoke test before the real run

Finding out at minute twenty that a collator or a metric argument is wrong is a bad way to spend an
afternoon. This builds a throwaway `Trainer` that runs twenty steps and stops.

The overrides are ordinary `TrainingArguments` fields, so they are set on a second arguments object
rather than threaded through `build_trainer` as keyword arguments. Anything the library accepts works
here, and the run above is left untouched.

In [ ]:
import gc

import torch


smoke_args = training_arguments(
    CLASSIFICATION,
    output_dir="/content/smoke",
    max_steps=20,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=10,
    save_steps=10,
    logging_steps=5,
)
smoke = build_trainer(model_args, data_args, smoke_args)
smoke.train()

del smoke
gc.collect()
torch.cuda.empty_cache()

## Fine-tune

In [ ]:
trainer.train()

## Learning curves

`trainer.state.log_history` is a flat list of dicts: training rows carry `loss`, evaluation rows carry
`eval_loss` and every key `compute_metrics` returned, prefixed with `eval_`. Splitting them apart with
`dropna` is enough to plot both.

In [ ]:
import pandas as pd


history = pd.DataFrame(trainer.state.log_history)
steps = history.dropna(subset=["loss"])
evals = history.dropna(subset=["eval_loss"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps["epoch"], steps["loss"], label="train", alpha=0.6)
axes[0].plot(evals["epoch"], evals["eval_loss"], marker="o", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(evals["epoch"], evals["eval_f1_weighted"], marker="o", label="weighted")
axes[1].plot(evals["epoch"], evals["eval_f1_macro"], marker="o", label="macro")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("epoch")
axes[1].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.show()

## Evaluate on the test split

Nothing in training or checkpoint selection has seen these rows, so this is the number to quote.

The gap between weighted and macro F1 here is the number worth reading. Weighted F1 is flattered by
the Neutral majority; macro F1 weights all three classes equally and tells you whether the model
actually learned Positive and Negative.


In [ ]:
import numpy as np

from headlines.bert.metrics import score_predictions


prediction = trainer.predict(trainer.test_dataset)
y_pred = np.argmax(prediction.predictions, axis=-1)
scores = score_predictions(prediction.label_ids, y_pred)

for name, value in scores.items():
    print(f"{name:<20} {value:.4f}")

### Confusion matrix

Rows are the weak label, columns are the prediction. The interesting cells are the off-diagonal ones
in the Neutral row and column — that is where a heuristic label and a learned model disagree most.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix


class_names = [ID2LABEL[index] for index in sorted(ID2LABEL)]
matrix = confusion_matrix(prediction.label_ids, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="viridis",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Validation split confusion matrix")
plt.show()

### Try it on new headlines

`build_trainer` passes `id2label` into `from_pretrained`, so the mapping travels with the model and
the pipeline prints `Positive` rather than `LABEL_2`.

In [ ]:
from transformers import pipeline


classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

samples = [
    "Record profits lift the FTSE to a new high",
    "Ministers face fresh criticism over delayed inquiry",
    "The committee will publish its findings on Tuesday",
]
for text, result in zip(samples, classifier(samples)):
    print(f"{result['label']:<9} {result['score']:.4f}  {text}")

## Save and download the checkpoint

`artifacts/` is gitignored, and a Colab runtime takes its disk with it when it disconnects. Download
the archive, or mount Drive and copy it there, before you close the tab.

In [ ]:
import shutil

from google.colab import files


trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

archive = shutil.make_archive("/content/modernbert-headlines", "zip", training_args.output_dir)
files.download(archive)